# 04 Data Model

> Change only `RAW` / `PROCESSED` paths if your folder location is different.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json

BASE = Path(r"C:\CHANGE\THIS\TO\YOUR\PROJECT")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

## Dimension and fact tables

In [5]:
# ==========================================
# 4. TRANSPORT FACT
# ==========================================

# Create date if it does not already exist
if "date" not in transport.columns:

    # Try to create date from arrival_time
    if "arrival_time" in transport.columns:
        transport["arrival_time"] = pd.to_datetime(
            transport["arrival_time"],
            errors="coerce",
            dayfirst=True
        )

        transport["date"] = transport["arrival_time"].dt.normalize()

    else:
        transport["date"] = pd.NaT


# Create distance_km if it does not already exist
if "distance_km" not in transport.columns:

    # Convert distance to number
    transport["distance"] = (
        pd.to_numeric(
            transport["distance"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"([-+]?\d*\.?\d+)")[0],
            errors="coerce"
        )
    )

    # Clean distance unit
    transport["distance_unit"] = (
        transport["distance_unit"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Start with original distance
    transport["distance_km"] = transport["distance"]

    # Miles → KM
    miles = transport["distance_unit"].isin(
        ["mile", "miles", "mi"]
    )

    transport.loc[miles, "distance_km"] = (
        transport.loc[miles, "distance"] * 1.60934
    )


# Make sure transit is numeric
transport["transit_hours"] = pd.to_numeric(
    transport["transit_hours"],
    errors="coerce"
)


# Remove invalid negative transit times
transport.loc[
    transport["transit_hours"] < 0,
    "transit_hours"
] = np.nan


# Vehicle number cleaning
if "vehicle_no" in transport.columns:
    transport["vehicle_no"] = (
        transport["vehicle_no"]
        .astype("string")
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )


# Create transport fact table
transport_fact = transport[
    [
        "trip_id",
        "date",
        "mandi_id",
        "destination_warehouse",
        "transit_hours",
        "distance_km",
        "vehicle_no",
        "driver_id"
    ]
].copy()


print("Transport Fact:", transport_fact.shape)

display(transport_fact.head())

Transport Fact: (10400, 8)


,trip_id,date,mandi_id,destination_warehouse,transit_hours,distance_km,vehicle_no,driver_id
0,TRP006374,2026-05-01,MANDI050,WH-Central,9.3,491.998625,<NA>,DRV264
1,TRP008869,2026-04-10,MANDI029,WH-West,NaN,263.500000,<NA>,DRV540
2,TRP000674,NaT,MANDI024,WH-South,6.2,276.500000,UP50BC6882,DRV722
3,TRP000036,NaT,MANDI-042,WH-Central,19.7,1070.700000,RJ52CD3274,DRV103
4,TRP008288,NaT,mandi_019,WH-South,17.1,728.900000,DL73DF1458,NaN


In [11]:
# ==========================================
# WEATHER DAILY + SAVE DATA MODEL
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

# ---------- DEFINE OUTPUT PATH ----------
P = Path("../data/processed")
P.mkdir(parents=True, exist_ok=True)

# ---------- WEATHER TIMESTAMP ----------

weather["timestamp"] = pd.to_datetime(
    weather["timestamp"],
    errors="coerce",
    utc=True
)

weather["timestamp_ist"] = weather["timestamp"].dt.tz_convert(
    "Asia/Kolkata"
)

weather["date"] = pd.to_datetime(
    weather["timestamp_ist"].dt.date,
    errors="coerce"
)

# ---------- TEMPERATURE ----------

if "temp_c" not in weather.columns:

    weather["temperature"] = pd.to_numeric(
        weather["temperature"]
        .astype(str)
        .str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

    weather["temp_unit"] = (
        weather["temp_unit"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    weather["temp_c"] = np.where(
        weather["temp_unit"].isin(
            ["f", "°f", "fahrenheit"]
        ),
        (weather["temperature"] - 32) * 5 / 9,
        weather["temperature"]
    )

else:

    weather["temp_c"] = pd.to_numeric(
        weather["temp_c"],
        errors="coerce"
    )

# ---------- RAINFALL ----------

if "rainfall_mm" not in weather.columns:

    weather["rainfall"] = pd.to_numeric(
        weather["rainfall"]
        .astype(str)
        .str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

    weather["rain_unit"] = (
        weather["rain_unit"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    weather["rainfall_mm"] = np.where(
        weather["rain_unit"].isin(
            ["in", "inch", "inches"]
        ),
        weather["rainfall"] * 25.4,
        weather["rainfall"]
    )

else:

    weather["rainfall_mm"] = pd.to_numeric(
        weather["rainfall_mm"],
        errors="coerce"
    )

# ---------- HUMIDITY ----------

if "humidity_percent" not in weather.columns:
    weather["humidity_percent"] = np.nan

weather["humidity_percent"] = pd.to_numeric(
    weather["humidity_percent"],
    errors="coerce"
)

# ---------- DAILY WEATHER ----------

weather_daily = (
    weather
    .dropna(subset=["date"])
    .groupby("date", as_index=False)
    .agg(
        temp_c=("temp_c", "mean"),
        rainfall_mm=("rainfall_mm", "sum"),
        humidity_percent=("humidity_percent", "mean")
    )
)

# ---------- SAVE DATA MODEL ----------

tables = {
    "mandi_dim": mandi_dim,
    "arrivals_fact": arrivals_fact,
    "prices_fact": prices_fact,
    "transport_fact": transport_fact,
    "weather_daily": weather_daily
}

for name, df in tables.items():
    df.to_csv(P / f"{name}.csv", index=False)

print("========================================")
print("DATA MODEL SAVED SUCCESSFULLY")
print("========================================")

for name, df in tables.items():
    print(f"{name}: {df.shape}")

print("\nWeather Daily:")
display(weather_daily.head())

DATA MODEL SAVED SUCCESSFULLY
mandi_dim: (57, 5)
arrivals_fact: (25750, 7)
prices_fact: (12000, 9)
transport_fact: (10400, 8)
weather_daily: (151, 4)

Weather Daily:


,date,temp_c,rainfall_mm,humidity_percent
0,2026-01-06,27.777778,-32.00,NaN
1,2026-01-07,35.550000,-11.13,60.0
2,2026-01-08,30.338889,59.90,66.0
3,2026-01-10,25.800000,7.20,50.0
4,2026-01-13,16.800000,21.70,36.0
